# COMP2024 Single-Notebook Colab Runner

This notebook is a single-file execution mirror of the modular coursework codebase.
Use it when you want upload-and-run convenience in Colab without relying on cross-file imports.


## Usage

1. Upload this notebook to Colab.
2. Ensure the two CSV files exist under `dataset/`.
3. Edit the configuration cell if you want to change budget, seeds, or results path.
4. Run cells from top to bottom.

Maintenance rule: update the modular `src/` implementation first, then regenerate or sync this notebook.


In [ ]:

from pathlib import Path
import yaml

PROJECT_ROOT = Path.cwd()
print('PROJECT_ROOT =', PROJECT_ROOT)

DEFAULT_CONFIG = {
    'dataset': {
        'train_csv': 'dataset/UNSW_NB15_training-set.csv',
        'test_csv': 'dataset/UNSW_NB15_testing-set.csv',
        'target': 'label',
        'drop_cols': ['id', 'attack_cat'],
    },
    'task': {
        'type': 'binary',
        'positive_label': 1,
        'negative_label': 0,
    },
    'protocol': {
        'val_size': 0.2,
    },
    'preprocess': {
        'categorical_cols': ['proto', 'service', 'state'],
        'numeric_impute': 'median',
        'onehot_handle_unknown': 'ignore',
    },
    'selection': {
        'mode': 'grouped_original_features',
        'k_min': 8,
    },
    'fitness': {
        'alpha_fpr': 0.05,
        'lambda_fpr': 20.0,
        'lambda_feat': 0.2,
        'metric_primary': 'recall',
    },
    'budget': {
        'evaluations_B': 2000,
        'seeds': list(range(20)),
    },
    'model': {
        'name': 'random_forest',
        'fixed': {
            'n_jobs': 1,
            'random_state_from_seed': True,
        },
        'search_space': {
            'n_estimators': [100, 600],
            'max_depth': [6, 30, None],
            'min_samples_split': [2, 20],
            'min_samples_leaf': [1, 8],
            'max_features': ['sqrt', 'log2'],
            'class_weight': [None, 'balanced'],
        },
    },
    'optimizers': {
        'ga': {
            'pop_size': 50,
            'generations': 40,
            'tournament_k': 3,
            'crossover_rate': 0.9,
            'mutation_rate': 0.05,
        },
        'pso': {
            'swarm_size': 50,
            'iterations': 40,
            'w': 0.7,
            'c1': 1.5,
            'c2': 1.5,
        },
        'sa': {
            'iterations': 2000,
            'T0': 1.0,
            'alpha': 0.995,
            'neighbor': {
                'bit_flips': [1, 3],
                'param_step_prob': 0.5,
            },
        },
    },
    'output': {
        'results_dir': 'results_single_notebook',
        'save_raw_per_seed': True,
        'save_plots': True,
    },
}

CONFIG = DEFAULT_CONFIG

RUN_SMOKE_TEST = True
SMOKE_BUDGET = 30
SMOKE_MAX_SEEDS = 1


## src/data.py


In [ ]:
"""Dataset loading utilities for UNSW-NB15 coursework experiments."""

from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import pandas as pd


@dataclass
class DatasetBundle:
    """Container for train/test data and resolved feature metadata."""

    train_df: pd.DataFrame
    test_df: pd.DataFrame
    target_col: str
    feature_cols: list[str]


def _resolve_target_col(configured_target: str | None, test_df: pd.DataFrame) -> str:
    if configured_target and configured_target in test_df.columns:
        return configured_target
    if "label" in test_df.columns:
        return "label"
    raise ValueError(
        "Unable to resolve target column. Set dataset.target in config "
        "to a valid column name present in testing CSV."
    )


def load_unsw_nb15(project_root: Path, config: dict[str, Any]) -> DatasetBundle:
    """Load train/test CSVs and resolve the exact feature columns."""
    dataset_cfg = config["dataset"]
    train_path = (project_root / dataset_cfg["train_csv"]).resolve()
    test_path = (project_root / dataset_cfg["test_csv"]).resolve()

    if not train_path.exists():
        raise FileNotFoundError(f"Training CSV not found: {train_path}")
    if not test_path.exists():
        raise FileNotFoundError(f"Testing CSV not found: {test_path}")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    target_col = _resolve_target_col(dataset_cfg.get("target"), test_df)
    if target_col not in train_df.columns:
        raise ValueError(f"Target column '{target_col}' not found in training CSV.")

    drop_cols = set(dataset_cfg.get("drop_cols", []))
    drop_cols.add(target_col)
    # Explicitly remove known leakage/non-feature columns before any preprocessing.

    feature_cols = [col for col in train_df.columns if col not in drop_cols]
    if not feature_cols:
        raise ValueError("No feature columns found after applying drop_cols/target.")

    missing_in_test = [col for col in feature_cols if col not in test_df.columns]
    if missing_in_test:
        raise ValueError(
            "Testing CSV is missing required feature columns: "
            + ", ".join(missing_in_test[:10])
        )

    return DatasetBundle(
        train_df=train_df,
        test_df=test_df,
        target_col=target_col,
        feature_cols=feature_cols,
    )


## src/preprocess.py


In [ ]:
"""Leakage-safe preprocessing fitted only on train folds."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder


def _build_one_hot_encoder(handle_unknown: str) -> OneHotEncoder:
    # Compatible across sklearn versions where sparse_output may not exist.
    try:
        return OneHotEncoder(handle_unknown=handle_unknown, sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown=handle_unknown, sparse=False)


@dataclass
class FittedPreprocessor:
    """Fitted preprocessor plus mapping from original to encoded indices."""

    transformer: ColumnTransformer
    original_features: list[str]
    numeric_features: list[str]
    categorical_features: list[str]
    group_to_indices: dict[str, list[int]]
    encoded_feature_names: list[str]

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        transformed = self.transformer.transform(df[self.original_features])
        if hasattr(transformed, "toarray"):
            transformed = transformed.toarray()
        return np.asarray(transformed, dtype=np.float32)


def fit_preprocessor(
    train_features_df: pd.DataFrame,
    original_features: list[str],
    categorical_cols: list[str],
    numeric_impute: str = "median",
    onehot_handle_unknown: str = "ignore",
) -> FittedPreprocessor:
    """Fit preprocessing on train-only data and return encoded mapping."""
    # Categorical membership is config-driven to keep feature grouping stable across runs.
    categorical = [c for c in categorical_cols if c in original_features]
    numeric = [c for c in original_features if c not in categorical]

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy=numeric_impute)),
        ]
    )
    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", _build_one_hot_encoder(onehot_handle_unknown)),
        ]
    )

    transformer = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric),
            ("cat", categorical_pipeline, categorical),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )
    # Caller controls leakage boundary; this fit must only see training fold data.
    transformer.fit(train_features_df[original_features])

    group_to_indices: dict[str, list[int]] = {}
    encoded_names: list[str] = []
    cursor = 0

    for col in numeric:
        group_to_indices[col] = [cursor]
        encoded_names.append(col)
        cursor += 1

    if categorical:
        ohe = transformer.named_transformers_["cat"].named_steps["onehot"]
        for col, categories in zip(categorical, ohe.categories_):
            width = len(categories)
            group_to_indices[col] = list(range(cursor, cursor + width))
            for cat in categories:
                encoded_names.append(f"{col}={cat}")
            cursor += width

    transformed_dim = transformer.transform(
        train_features_df.iloc[[0]][original_features]
    ).shape[1]
    if cursor != transformed_dim:
        raise RuntimeError(
            f"Encoded index mapping mismatch. mapped={cursor}, actual={transformed_dim}"
        )

    return FittedPreprocessor(
        transformer=transformer,
        original_features=original_features,
        numeric_features=numeric,
        categorical_features=categorical,
        group_to_indices=group_to_indices,
        encoded_feature_names=encoded_names,
    )


def split_xy(
    df: pd.DataFrame,
    feature_cols: list[str],
    target_col: str,
) -> tuple[pd.DataFrame, np.ndarray]:
    """Extract X dataframe and y array (int) from a dataframe."""
    x_df = df[feature_cols].copy()
    y = df[target_col].astype(int).to_numpy()
    return x_df, y


## src/metrics.py


In [ ]:
"""Metric helpers for IDS binary classification."""

from __future__ import annotations

from typing import Any

import numpy as np
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score


def compute_binary_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    selected_features: int,
    total_features: int,
    runtime_sec: float,
    fit_time_sec: float,
    predict_time_sec: float,
) -> dict[str, Any]:
    """Compute metrics required by coursework rubric and optimization protocol."""
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    else:
        tn = fp = fn = tp = 0

    # FPR is central to IDS trade-offs; keep explicit and robust for edge cases.
    denom = fp + tn
    fpr = float(fp / denom) if denom > 0 else 0.0

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "fpr": fpr,
        "selected_features": int(selected_features),
        "selected_feature_ratio": float(selected_features / max(total_features, 1)),
        "runtime_sec": float(runtime_sec),
        "fit_time_sec": float(fit_time_sec),
        "predict_time_sec": float(predict_time_sec),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


## src/representation.py


In [ ]:
"""Solution representation and decoding for feature+hyperparameter search."""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any

import numpy as np


PARAM_GENE_COUNT = 6


@dataclass(frozen=True)
class SearchSpace:
    n_estimators_min: int
    n_estimators_max: int
    max_depth_min: int
    max_depth_max: int
    allow_none_depth: bool
    min_samples_split_min: int
    min_samples_split_max: int
    min_samples_leaf_min: int
    min_samples_leaf_max: int
    max_features_options: list[str]
    class_weight_options: list[str | None]


@dataclass
class CandidateSolution:
    mask: np.ndarray
    params: dict[str, Any]

    @property
    def k(self) -> int:
        return int(self.mask.sum())


def vector_dim(n_features: int) -> int:
    return n_features + PARAM_GENE_COUNT


def random_vector(rng: np.random.Generator, n_features: int) -> np.ndarray:
    return rng.random(vector_dim(n_features), dtype=np.float64)


def clamp_vector(vec: np.ndarray) -> np.ndarray:
    return np.clip(vec, 0.0, 1.0)


def build_search_space(model_cfg: dict[str, Any]) -> SearchSpace:
    search_cfg = model_cfg["search_space"]
    depth_cfg = search_cfg["max_depth"]
    allow_none_depth = any(v is None for v in depth_cfg)

    class_weight_options: list[str | None] = []
    for item in search_cfg["class_weight"]:
        class_weight_options.append(item)

    return SearchSpace(
        n_estimators_min=int(search_cfg["n_estimators"][0]),
        n_estimators_max=int(search_cfg["n_estimators"][1]),
        max_depth_min=int(depth_cfg[0]),
        max_depth_max=int(depth_cfg[1]),
        allow_none_depth=allow_none_depth,
        min_samples_split_min=int(search_cfg["min_samples_split"][0]),
        min_samples_split_max=int(search_cfg["min_samples_split"][1]),
        min_samples_leaf_min=int(search_cfg["min_samples_leaf"][0]),
        min_samples_leaf_max=int(search_cfg["min_samples_leaf"][1]),
        max_features_options=[str(v) for v in search_cfg["max_features"]],
        class_weight_options=class_weight_options,
    )


def _decode_int(gene: float, lower: int, upper: int) -> int:
    return int(round(lower + gene * (upper - lower)))


def _decode_choice(gene: float, choices: list[Any]) -> Any:
    idx = min(int(gene * len(choices)), len(choices) - 1)
    return choices[idx]


def _enforce_k_min(mask_scores: np.ndarray, mask: np.ndarray, k_min: int) -> np.ndarray:
    enforced = mask.copy()
    # Keep at least k_min original features so optimizers cannot exploit trivial empty masks.
    need = max(1, min(k_min, mask_scores.shape[0]))
    if int(enforced.sum()) >= need:
        return enforced
    top_idx = np.argsort(mask_scores)[-need:]
    enforced[:] = False
    enforced[top_idx] = True
    return enforced


def decode_solution(
    vector: np.ndarray,
    n_features: int,
    k_min: int,
    search_space: SearchSpace,
) -> CandidateSolution:
    """Decode [0,1] genes into feature mask + RF parameter dictionary."""
    vec = clamp_vector(vector)
    mask_scores = vec[:n_features]
    mask = mask_scores >= 0.5
    mask = _enforce_k_min(mask_scores, mask, k_min)

    # Parameter genes are shared across all optimizers, so decoding must stay deterministic.
    g = vec[n_features : n_features + PARAM_GENE_COUNT]

    depth_choices: list[int | None] = list(
        range(search_space.max_depth_min, search_space.max_depth_max + 1)
    )
    if search_space.allow_none_depth:
        depth_choices = [None] + depth_choices

    params: dict[str, Any] = {
        "n_estimators": _decode_int(
            g[0], search_space.n_estimators_min, search_space.n_estimators_max
        ),
        "max_depth": _decode_choice(g[1], depth_choices),
        "min_samples_split": _decode_int(
            g[2], search_space.min_samples_split_min, search_space.min_samples_split_max
        ),
        "min_samples_leaf": _decode_int(
            g[3], search_space.min_samples_leaf_min, search_space.min_samples_leaf_max
        ),
        "max_features": _decode_choice(g[4], search_space.max_features_options),
        "class_weight": _decode_choice(g[5], search_space.class_weight_options),
    }

    return CandidateSolution(mask=mask.astype(bool), params=params)


def solution_key(solution: CandidateSolution) -> str:
    mask_bits = "".join("1" if bit else "0" for bit in solution.mask.tolist())
    ordered_params = [
        str(solution.params["n_estimators"]),
        str(solution.params["max_depth"]),
        str(solution.params["min_samples_split"]),
        str(solution.params["min_samples_leaf"]),
        str(solution.params["max_features"]),
        str(solution.params["class_weight"]),
    ]
    return f"{mask_bits}|{'|'.join(ordered_params)}"


def mask_to_encoded_indices(
    mask: np.ndarray,
    original_features: list[str],
    group_to_indices: dict[str, list[int]],
) -> np.ndarray:
    indices: list[int] = []
    for is_selected, feature in zip(mask.tolist(), original_features):
        if is_selected:
            # Grouped selection: selecting one original categorical feature enables all its one-hot columns.
            indices.extend(group_to_indices[feature])
    return np.asarray(indices, dtype=np.int32)


## src/evaluator.py


In [ ]:
"""Objective evaluator with per-(algorithm,seed) cache isolation."""

from __future__ import annotations

import time
from dataclasses import dataclass
from typing import Any

import numpy as np
from sklearn.ensemble import RandomForestClassifier

    CandidateSolution,
    SearchSpace,
    decode_solution,
    mask_to_encoded_indices,
    solution_key,
)


@dataclass
class EvaluationRecord:
    score: float
    metrics: dict[str, Any]
    solution: CandidateSolution
    cache_hit: bool


def _build_rf(
    params: dict[str, Any],
    seed: int,
    n_jobs: int,
) -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=int(params["n_estimators"]),
        max_depth=params["max_depth"],
        min_samples_split=int(params["min_samples_split"]),
        min_samples_leaf=int(params["min_samples_leaf"]),
        max_features=params["max_features"],
        class_weight=params["class_weight"],
        random_state=seed,
        n_jobs=n_jobs,
    )


def compute_fitness(
    recall: float,
    fpr: float,
    k: int,
    d: int,
    alpha: float,
    lambda_fpr: float,
    lambda_feat: float,
) -> float:
    return float(
        recall - lambda_fpr * max(0.0, fpr - alpha) - lambda_feat * (k / max(d, 1))
    )


class ObjectiveEvaluator:
    """Validation-only evaluator used by metaheuristics."""

    def __init__(
        self,
        algorithm_name: str,
        seed: int,
        x_train_inner: np.ndarray,
        y_train_inner: np.ndarray,
        x_val_inner: np.ndarray,
        y_val_inner: np.ndarray,
        original_features: list[str],
        group_to_indices: dict[str, list[int]],
        search_space: SearchSpace,
        k_min: int,
        fitness_cfg: dict[str, Any],
        n_jobs: int = 1,
    ) -> None:
        self.algorithm_name = algorithm_name
        self.seed = seed
        self.x_train_inner = x_train_inner
        self.y_train_inner = y_train_inner
        self.x_val_inner = x_val_inner
        self.y_val_inner = y_val_inner
        self.original_features = original_features
        self.group_to_indices = group_to_indices
        self.search_space = search_space
        self.k_min = k_min
        self.n_jobs = n_jobs
        self.alpha = float(fitness_cfg["alpha_fpr"])
        self.lambda_fpr = float(fitness_cfg["lambda_fpr"])
        self.lambda_feat = float(fitness_cfg["lambda_feat"])
        self.total_features = len(original_features)
        # Cache is isolated per (algorithm, seed) by object lifetime.
        self._cache: dict[str, tuple[float, dict[str, Any], CandidateSolution]] = {}

    def evaluate(self, vector: np.ndarray) -> EvaluationRecord:
        solution = decode_solution(
            vector=vector,
            n_features=self.total_features,
            k_min=self.k_min,
            search_space=self.search_space,
        )
        key = solution_key(solution)
        if key in self._cache:
            # Cache hit still counts as an optimizer evaluation in caller logic; only model fitting is skipped.
            cached_score, cached_metrics, cached_solution = self._cache[key]
            return EvaluationRecord(
                score=cached_score,
                metrics=dict(cached_metrics),
                solution=CandidateSolution(
                    mask=cached_solution.mask.copy(),
                    params=dict(cached_solution.params),
                ),
                cache_hit=True,
            )

        selected_indices = mask_to_encoded_indices(
            mask=solution.mask,
            original_features=self.original_features,
            group_to_indices=self.group_to_indices,
        )
        if selected_indices.size == 0:
            raise RuntimeError("Decoded empty feature set. k_min enforcement failed.")

        # Validation-only objective evaluation (test set is never touched here).
        model = _build_rf(solution.params, seed=self.seed, n_jobs=self.n_jobs)
        start = time.perf_counter()
        fit_start = time.perf_counter()
        model.fit(self.x_train_inner[:, selected_indices], self.y_train_inner)
        fit_time = time.perf_counter() - fit_start

        pred_start = time.perf_counter()
        y_pred = model.predict(self.x_val_inner[:, selected_indices])
        pred_time = time.perf_counter() - pred_start
        runtime = time.perf_counter() - start

        metrics = compute_binary_metrics(
            y_true=self.y_val_inner,
            y_pred=y_pred,
            selected_features=solution.k,
            total_features=self.total_features,
            runtime_sec=runtime,
            fit_time_sec=fit_time,
            predict_time_sec=pred_time,
        )
        score = compute_fitness(
            recall=metrics["recall"],
            fpr=metrics["fpr"],
            k=solution.k,
            d=self.total_features,
            alpha=self.alpha,
            lambda_fpr=self.lambda_fpr,
            lambda_feat=self.lambda_feat,
        )

        self._cache[key] = (
            score,
            dict(metrics),
            CandidateSolution(mask=solution.mask.copy(), params=dict(solution.params)),
        )
        return EvaluationRecord(score=score, metrics=metrics, solution=solution, cache_hit=False)


def evaluate_solution_on_test(
    solution: CandidateSolution,
    x_train_full: np.ndarray,
    y_train_full: np.ndarray,
    x_test: np.ndarray,
    y_test: np.ndarray,
    original_features: list[str],
    group_to_indices: dict[str, list[int]],
    fitness_cfg: dict[str, Any],
    seed: int,
    n_jobs: int = 1,
) -> EvaluationRecord:
    """Final one-shot test evaluation performed after optimization."""
    # This function is intentionally isolated from optimization to preserve test-set integrity.
    selected_indices = mask_to_encoded_indices(
        mask=solution.mask,
        original_features=original_features,
        group_to_indices=group_to_indices,
    )
    if selected_indices.size == 0:
        raise RuntimeError("Attempted test evaluation with empty selected feature set.")

    model = _build_rf(solution.params, seed=seed, n_jobs=n_jobs)
    start = time.perf_counter()
    fit_start = time.perf_counter()
    model.fit(x_train_full[:, selected_indices], y_train_full)
    fit_time = time.perf_counter() - fit_start
    pred_start = time.perf_counter()
    y_pred = model.predict(x_test[:, selected_indices])
    pred_time = time.perf_counter() - pred_start
    runtime = time.perf_counter() - start

    total_features = len(original_features)
    metrics = compute_binary_metrics(
        y_true=y_test,
        y_pred=y_pred,
        selected_features=solution.k,
        total_features=total_features,
        runtime_sec=runtime,
        fit_time_sec=fit_time,
        predict_time_sec=pred_time,
    )
    score = compute_fitness(
        recall=metrics["recall"],
        fpr=metrics["fpr"],
        k=solution.k,
        d=total_features,
        alpha=float(fitness_cfg["alpha_fpr"]),
        lambda_fpr=float(fitness_cfg["lambda_fpr"]),
        lambda_feat=float(fitness_cfg["lambda_feat"]),
    )
    return EvaluationRecord(score=score, metrics=metrics, solution=solution, cache_hit=False)


## src/baseline.py


In [ ]:
"""Baseline Random Forest evaluation (all features, default hyperparameters)."""

from __future__ import annotations

import numpy as np



def build_baseline_solution(n_original_features: int) -> CandidateSolution:
    return CandidateSolution(
        mask=np.ones(n_original_features, dtype=bool),
        params={
            # Baseline must stay non-metaheuristic with default RF settings for fair benchmarking.
            "n_estimators": 100,
            "max_depth": None,
            "min_samples_split": 2,
            "min_samples_leaf": 1,
            "max_features": "sqrt",
            "class_weight": None,
        },
    )


def run_baseline_test(
    seed: int,
    x_train_full,
    y_train_full,
    x_test,
    y_test,
    original_features,
    group_to_indices,
    fitness_cfg,
) -> EvaluationRecord:
    solution = build_baseline_solution(len(original_features))
    return evaluate_solution_on_test(
        solution=solution,
        x_train_full=x_train_full,
        y_train_full=y_train_full,
        x_test=x_test,
        y_test=y_test,
        original_features=original_features,
        group_to_indices=group_to_indices,
        fitness_cfg=fitness_cfg,
        seed=seed,
        n_jobs=1,
    )


## src/optimizers/ga.py


In [ ]:
"""Genetic Algorithm optimizer with fixed objective-evaluation budget."""

from __future__ import annotations

from typing import Any

import numpy as np



def _tournament_index(
    scores: np.ndarray, k: int, rng: np.random.Generator
) -> int:
    candidates = rng.integers(low=0, high=len(scores), size=max(2, k))
    best = candidates[0]
    for idx in candidates[1:]:
        if scores[idx] > scores[best]:
            best = idx
    return int(best)


def _crossover_and_mutate(
    parent_a: np.ndarray,
    parent_b: np.ndarray,
    n_feature_genes: int,
    crossover_rate: float,
    mutation_rate: float,
    rng: np.random.Generator,
) -> np.ndarray:
    child = parent_a.copy()
    if rng.random() < crossover_rate:
        cut = int(rng.integers(1, max(2, n_feature_genes)))
        child[:n_feature_genes] = np.concatenate(
            [parent_a[:cut], parent_b[cut:n_feature_genes]]
        )
        for idx in range(n_feature_genes, len(child)):
            child[idx] = parent_a[idx] if rng.random() < 0.5 else parent_b[idx]

    for idx in range(n_feature_genes):
        if rng.random() < mutation_rate:
            child[idx] = 1.0 - child[idx]
    for idx in range(n_feature_genes, len(child)):
        if rng.random() < mutation_rate:
            child[idx] += rng.normal(loc=0.0, scale=0.12)
    return clamp_vector(child)


def run_ga(
    evaluator,
    budget_b: int,
    n_feature_genes: int,
    rng: np.random.Generator,
    ga_cfg: dict[str, Any],
) -> dict[str, Any]:
    """Run GA; every call to evaluator consumes one budget unit (cache hit included)."""
    pop_size = int(ga_cfg.get("pop_size", 30))
    tournament_k = int(ga_cfg.get("tournament_k", 3))
    crossover_rate = float(ga_cfg.get("crossover_rate", 0.9))
    mutation_rate = float(ga_cfg.get("mutation_rate", 0.05))

    population = np.vstack([random_vector(rng, n_feature_genes) for _ in range(pop_size)])
    scores = np.full(pop_size, -np.inf, dtype=np.float64)
    best_score = -np.inf
    best_solution = None
    best_metrics = None
    best_vector = None
    history: list[dict[str, Any]] = []

    evaluations = 0
    # Evaluate initial population first so GA and other methods pay comparable startup cost.
    for i in range(pop_size):
        if evaluations >= budget_b:
            break
        rec = evaluator.evaluate(population[i])
        scores[i] = rec.score
        evaluations += 1
        if rec.score > best_score:
            best_score = rec.score
            best_solution = rec.solution
            best_metrics = dict(rec.metrics)
            best_vector = population[i].copy()
        history.append(
            {
                "evaluation": evaluations,
                "score": rec.score,
                "best_score": best_score,
                "cache_hit": int(rec.cache_hit),
            }
        )

    while evaluations < budget_b:
        idx_a = _tournament_index(scores, tournament_k, rng)
        idx_b = _tournament_index(scores, tournament_k, rng)
        child = _crossover_and_mutate(
            parent_a=population[idx_a],
            parent_b=population[idx_b],
            n_feature_genes=n_feature_genes,
            crossover_rate=crossover_rate,
            mutation_rate=mutation_rate,
            rng=rng,
        )
        rec = evaluator.evaluate(child)
        evaluations += 1

        # Steady-state replacement keeps budget accounting simple: one new evaluation per loop.
        worst_idx = int(np.argmin(scores))
        population[worst_idx] = child
        scores[worst_idx] = rec.score

        if rec.score > best_score:
            best_score = rec.score
            best_solution = rec.solution
            best_metrics = dict(rec.metrics)
            best_vector = child.copy()

        history.append(
            {
                "evaluation": evaluations,
                "score": rec.score,
                "best_score": best_score,
                "cache_hit": int(rec.cache_hit),
            }
        )

    return {
        "best_score": float(best_score),
        "best_solution": best_solution,
        "best_metrics": best_metrics,
        "best_vector": best_vector,
        "history": history,
        "evaluations": evaluations,
    }


## src/optimizers/pso.py


In [ ]:
"""Hybrid Binary/Continuous PSO with fixed objective-evaluation budget."""

from __future__ import annotations

from typing import Any

import numpy as np



def _sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(x, -30.0, 30.0)))


def run_pso(
    evaluator,
    budget_b: int,
    n_feature_genes: int,
    rng: np.random.Generator,
    pso_cfg: dict[str, Any],
) -> dict[str, Any]:
    """Run PSO; cache hits still count as evaluations."""
    swarm_size = int(pso_cfg.get("swarm_size", 30))
    w = float(pso_cfg.get("w", 0.7))
    c1 = float(pso_cfg.get("c1", 1.5))
    c2 = float(pso_cfg.get("c2", 1.5))

    dim = n_feature_genes + 6
    positions = np.vstack([random_vector(rng, n_feature_genes) for _ in range(swarm_size)])
    velocities = rng.normal(loc=0.0, scale=0.2, size=(swarm_size, dim))
    pbest_positions = positions.copy()
    pbest_scores = np.full(swarm_size, -np.inf, dtype=np.float64)

    best_score = -np.inf
    best_solution = None
    best_metrics = None
    best_vector = None
    history: list[dict[str, Any]] = []
    evaluations = 0

    for i in range(swarm_size):
        if evaluations >= budget_b:
            break
        rec = evaluator.evaluate(positions[i])
        pbest_scores[i] = rec.score
        evaluations += 1
        if rec.score > best_score:
            best_score = rec.score
            best_solution = rec.solution
            best_metrics = dict(rec.metrics)
            best_vector = positions[i].copy()
        history.append(
            {
                "evaluation": evaluations,
                "score": rec.score,
                "best_score": best_score,
                "cache_hit": int(rec.cache_hit),
            }
        )

    if best_vector is None:
        raise RuntimeError("PSO failed to initialize best vector.")

    while evaluations < budget_b:
        for i in range(swarm_size):
            if evaluations >= budget_b:
                break

            r1 = rng.random(dim)
            r2 = rng.random(dim)
            velocities[i] = (
                w * velocities[i]
                + c1 * r1 * (pbest_positions[i] - positions[i])
                + c2 * r2 * (best_vector - positions[i])
            )

            proposal = positions[i] + velocities[i]
            proposal = clamp_vector(proposal)

            # Binary update for feature genes and continuous update for hyperparameter genes.
            mask_probs = _sigmoid(velocities[i, :n_feature_genes])
            proposal[:n_feature_genes] = (
                rng.random(n_feature_genes) < mask_probs
            ).astype(np.float64)

            positions[i] = proposal
            rec = evaluator.evaluate(positions[i])
            evaluations += 1

            if rec.score > pbest_scores[i]:
                pbest_scores[i] = rec.score
                pbest_positions[i] = positions[i].copy()
            if rec.score > best_score:
                best_score = rec.score
                best_solution = rec.solution
                best_metrics = dict(rec.metrics)
                best_vector = positions[i].copy()

            history.append(
                {
                    "evaluation": evaluations,
                    "score": rec.score,
                    "best_score": best_score,
                    "cache_hit": int(rec.cache_hit),
                }
            )

    return {
        "best_score": float(best_score),
        "best_solution": best_solution,
        "best_metrics": best_metrics,
        "best_vector": best_vector,
        "history": history,
        "evaluations": evaluations,
    }


## src/optimizers/sa.py


In [ ]:
"""Simulated Annealing optimizer with fixed objective-evaluation budget."""

from __future__ import annotations

import math
from typing import Any

import numpy as np



def _propose_neighbor(
    current: np.ndarray,
    n_feature_genes: int,
    rng: np.random.Generator,
    bit_flips_range: tuple[int, int],
    param_step_prob: float,
    temperature_ratio: float,
) -> np.ndarray:
    neighbor = current.copy()

    lo, hi = bit_flips_range
    n_flips = int(rng.integers(low=lo, high=hi + 1))
    flip_idx = rng.choice(n_feature_genes, size=min(n_flips, n_feature_genes), replace=False)
    neighbor[flip_idx] = 1.0 - neighbor[flip_idx]

    step_scale = 0.12 * max(temperature_ratio, 0.1)
    for idx in range(n_feature_genes, len(neighbor)):
        if rng.random() < param_step_prob:
            neighbor[idx] += rng.normal(loc=0.0, scale=step_scale)
    return clamp_vector(neighbor)


def run_sa(
    evaluator,
    budget_b: int,
    n_feature_genes: int,
    rng: np.random.Generator,
    sa_cfg: dict[str, Any],
) -> dict[str, Any]:
    """Run SA; each evaluator call consumes one budget unit."""
    t0 = float(sa_cfg.get("T0", 1.0))
    cooling = float(sa_cfg.get("alpha", 0.995))
    neighbor_cfg = sa_cfg.get("neighbor", {})
    flips = neighbor_cfg.get("bit_flips", [1, 3])
    bit_flips_range = (int(flips[0]), int(flips[1]))
    param_step_prob = float(neighbor_cfg.get("param_step_prob", 0.5))

    current = random_vector(rng, n_feature_genes)
    rec_current = evaluator.evaluate(current)
    evaluations = 1
    current_score = rec_current.score
    current_solution = rec_current.solution

    best_score = current_score
    best_solution = current_solution
    best_metrics = dict(rec_current.metrics)
    best_vector = current.copy()
    history: list[dict[str, Any]] = [
        {
            "evaluation": evaluations,
            "score": current_score,
            "best_score": best_score,
            "cache_hit": int(rec_current.cache_hit),
        }
    ]

    step = 0
    while evaluations < budget_b:
        step += 1
        temperature = t0 * (cooling**step)
        proposal = _propose_neighbor(
            current=current,
            n_feature_genes=n_feature_genes,
            rng=rng,
            bit_flips_range=bit_flips_range,
            param_step_prob=param_step_prob,
            temperature_ratio=temperature / max(t0, 1e-9),
        )
        rec_proposal = evaluator.evaluate(proposal)
        evaluations += 1

        delta = rec_proposal.score - current_score
        if delta >= 0:
            accept = True
        else:
            # Metropolis acceptance allows occasional worse moves to escape local minima.
            acceptance_prob = math.exp(delta / max(temperature, 1e-9))
            accept = rng.random() < acceptance_prob

        if accept:
            current = proposal
            current_score = rec_proposal.score
            current_solution = rec_proposal.solution

        if rec_proposal.score > best_score:
            best_score = rec_proposal.score
            best_solution = rec_proposal.solution
            best_metrics = dict(rec_proposal.metrics)
            best_vector = proposal.copy()

        history.append(
            {
                "evaluation": evaluations,
                "score": rec_proposal.score,
                "best_score": best_score,
                "cache_hit": int(rec_proposal.cache_hit),
            }
        )

    return {
        "best_score": float(best_score),
        "best_solution": best_solution,
        "best_metrics": best_metrics,
        "best_vector": best_vector,
        "history": history,
        "evaluations": evaluations,
    }


## src/runner.py


In [ ]:
"""Experiment runner orchestrating leakage-safe protocol and outputs."""

from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.model_selection import train_test_split



def load_experiment_config(config_path: Path) -> dict[str, Any]:
    if not config_path.exists():
        raise FileNotFoundError(f"Config file not found: {config_path}")
    with config_path.open("r", encoding="utf-8") as fh:
        config = yaml.safe_load(fh)

    required_keys = [
        "dataset",
        "preprocess",
        "selection",
        "fitness",
        "budget",
        "model",
        "optimizers",
        "output",
    ]
    for key in required_keys:
        if key not in config:
            raise ValueError(f"Missing required config section: {key}")
    return config


def _ensure_dirs(base_results_dir: Path) -> dict[str, Path]:
    raw_dir = base_results_dir / "raw"
    convergence_dir = base_results_dir / "convergence"
    plots_dir = base_results_dir / "plots"
    for d in [base_results_dir, raw_dir, convergence_dir, plots_dir]:
        d.mkdir(parents=True, exist_ok=True)
    return {"results": base_results_dir, "raw": raw_dir, "convergence": convergence_dir, "plots": plots_dir}


def _save_run_repro_artifacts(
    out_dirs: dict[str, Path],
    config: dict[str, Any],
    seeds: list[int],
    config_raw_text: str | None = None,
) -> None:
    # Persist exact run inputs so results can be reproduced without guessing runtime flags.
    run_config_path = out_dirs["results"] / "run_config.yaml"
    if config_raw_text is not None:
        run_config_path.write_text(config_raw_text, encoding="utf-8")
    else:
        with run_config_path.open("w", encoding="utf-8") as fh:
            yaml.safe_dump(config, fh, sort_keys=False)

    seed_list_path = out_dirs["results"] / "seed_list.txt"
    seed_list_path.write_text("\n".join(str(s) for s in seeds) + "\n", encoding="utf-8")


def _append_row_csv(path: Path, row: dict[str, Any]) -> None:
    # Flush each completed result row to disk to survive Colab/runtime interruptions.
    df = pd.DataFrame([row])
    df.to_csv(path, mode="a", index=False, header=not path.exists())


def _optimizer_rng(seed: int, algorithm: str) -> np.random.Generator:
    offsets = {"ga": 11, "pso": 23, "sa": 37}
    return np.random.default_rng(seed * 10_000 + offsets[algorithm])


def _save_convergence(history: list[dict[str, Any]], out_path: Path, algorithm: str, seed: int) -> None:
    if not history:
        return
    hist_df = pd.DataFrame(history)
    hist_df["algorithm"] = algorithm
    hist_df["seed"] = seed
    hist_df.to_csv(out_path, index=False)


def _aggregate_summary(all_runs: pd.DataFrame, out_path: Path) -> pd.DataFrame:
    numeric_cols = [
        "val_best_score",
        "test_score",
        "test_accuracy",
        "test_precision",
        "test_recall",
        "test_f1",
        "test_fpr",
        "test_selected_features",
        "test_runtime_sec",
        "test_fit_time_sec",
        "test_predict_time_sec",
    ]
    grouped = all_runs.groupby("algorithm")[numeric_cols].agg(["mean", "std"]).reset_index()
    grouped.columns = [
        "algorithm" if col[0] == "algorithm" else f"{col[0]}_{col[1]}"
        for col in grouped.columns
    ]
    grouped.to_csv(out_path, index=False)
    return grouped


def _plot_box(
    all_runs: pd.DataFrame,
    metric_col: str,
    y_label: str,
    out_path: Path,
) -> None:
    ordered_algorithms = sorted(all_runs["algorithm"].unique().tolist())
    values = [
        all_runs.loc[all_runs["algorithm"] == alg, metric_col].to_numpy()
        for alg in ordered_algorithms
    ]
    plt.figure(figsize=(10, 5))
    plt.boxplot(values, labels=ordered_algorithms, showmeans=True)
    plt.ylabel(y_label)
    plt.title(f"{y_label} by algorithm")
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


def _plot_convergence(convergence_dir: Path, out_path: Path) -> None:
    files = sorted(convergence_dir.glob("*.csv"))
    if not files:
        return
    frames = [pd.read_csv(p) for p in files]
    conv_df = pd.concat(frames, ignore_index=True)

    plt.figure(figsize=(10, 5))
    for alg, sub in conv_df.groupby("algorithm"):
        mean_curve = sub.groupby("evaluation")["best_score"].mean().sort_index()
        plt.plot(mean_curve.index, mean_curve.values, label=alg)
    plt.xlabel("Objective evaluations")
    plt.ylabel("Best validation fitness")
    plt.title("Mean convergence by algorithm")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_path, dpi=180)
    plt.close()


def _create_plots(all_runs: pd.DataFrame, convergence_dir: Path, plots_dir: Path) -> None:
    _plot_box(all_runs, "test_recall", "Test Recall", plots_dir / "box_test_recall.png")
    _plot_box(all_runs, "test_fpr", "Test FPR", plots_dir / "box_test_fpr.png")
    _plot_box(
        all_runs,
        "test_selected_features",
        "Selected Original Features",
        plots_dir / "box_selected_features.png",
    )
    _plot_convergence(convergence_dir, plots_dir / "convergence_mean_best_score.png")


def _run_single_optimizer(
    algorithm: str,
    evaluator: ObjectiveEvaluator,
    budget_b: int,
    n_feature_genes: int,
    seed: int,
    optimizer_cfg: dict[str, Any],
) -> dict[str, Any]:
    # All optimizers consume the same objective-evaluation budget for fair comparison.
    rng = _optimizer_rng(seed=seed, algorithm=algorithm)
    if algorithm == "ga":
        return run_ga(
            evaluator=evaluator,
            budget_b=budget_b,
            n_feature_genes=n_feature_genes,
            rng=rng,
            ga_cfg=optimizer_cfg,
        )
    if algorithm == "pso":
        return run_pso(
            evaluator=evaluator,
            budget_b=budget_b,
            n_feature_genes=n_feature_genes,
            rng=rng,
            pso_cfg=optimizer_cfg,
        )
    if algorithm == "sa":
        return run_sa(
            evaluator=evaluator,
            budget_b=budget_b,
            n_feature_genes=n_feature_genes,
            rng=rng,
            sa_cfg=optimizer_cfg,
        )
    raise ValueError(f"Unknown algorithm: {algorithm}")


def run_coursework_experiment(
    project_root: Path,
    config: dict[str, Any],
    budget_override: int | None = None,
    max_seeds: int | None = None,
    skip_plots: bool = False,
    config_raw_text: str | None = None,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    data: DatasetBundle = load_unsw_nb15(project_root=project_root, config=config)
    feature_cols = data.feature_cols
    d_features = len(feature_cols)
    if d_features <= 0:
        raise RuntimeError("No usable feature columns resolved.")

    budget_b = int(budget_override or config["budget"]["evaluations_B"])
    if budget_b <= 0:
        raise ValueError("Evaluation budget B must be > 0.")

    seeds: list[int] = [int(s) for s in config["budget"]["seeds"]]
    if max_seeds is not None:
        seeds = seeds[:max_seeds]

    out_dirs = _ensure_dirs(project_root / config["output"]["results_dir"])
    _save_run_repro_artifacts(
        out_dirs=out_dirs,
        config=config,
        seeds=seeds,
        config_raw_text=config_raw_text,
    )
    incremental_path = out_dirs["raw"] / "all_runs_incremental.csv"
    if incremental_path.exists():
        incremental_path.unlink()

    search_space: SearchSpace = build_search_space(config["model"])
    k_min = int(config["selection"]["k_min"])
    val_size = float(config.get("protocol", {}).get("val_size", 0.2))

    all_rows: list[dict[str, Any]] = []
    categorical_cols = list(config["preprocess"].get("categorical_cols", []))
    numeric_impute = str(config["preprocess"].get("numeric_impute", "median"))
    onehot_handle_unknown = str(config["preprocess"].get("onehot_handle_unknown", "ignore"))

    # Full train/test arrays are built once per seed to keep seed-specific RF randomness.
    full_x_df, full_y = split_xy(data.train_df, feature_cols, data.target_col)
    test_x_df, test_y = split_xy(data.test_df, feature_cols, data.target_col)

    for seed in seeds:
        # Split only training CSV into inner train/validation for model selection (no test leakage).
        train_inner_idx, val_inner_idx = train_test_split(
            np.arange(len(data.train_df)),
            test_size=val_size,
            random_state=seed,
            stratify=full_y,
        )
        train_inner_df = data.train_df.iloc[train_inner_idx].reset_index(drop=True)
        val_inner_df = data.train_df.iloc[val_inner_idx].reset_index(drop=True)
        train_inner_x_df, y_train_inner = split_xy(train_inner_df, feature_cols, data.target_col)
        val_inner_x_df, y_val_inner = split_xy(val_inner_df, feature_cols, data.target_col)

        # Leakage-safe: fit preprocessing ONLY on train_inner for optimization.
        pre_inner = fit_preprocessor(
            train_features_df=train_inner_x_df,
            original_features=feature_cols,
            categorical_cols=categorical_cols,
            numeric_impute=numeric_impute,
            onehot_handle_unknown=onehot_handle_unknown,
        )
        x_train_inner = pre_inner.transform(train_inner_x_df)
        x_val_inner = pre_inner.transform(val_inner_x_df)

        # Final one-shot evaluation uses preprocessing fit on full training data.
        pre_full = fit_preprocessor(
            train_features_df=full_x_df,
            original_features=feature_cols,
            categorical_cols=categorical_cols,
            numeric_impute=numeric_impute,
            onehot_handle_unknown=onehot_handle_unknown,
        )
        x_train_full = pre_full.transform(full_x_df)
        x_test = pre_full.transform(test_x_df)

        baseline = run_baseline_test(
            seed=seed,
            x_train_full=x_train_full,
            y_train_full=full_y,
            x_test=x_test,
            y_test=test_y,
            original_features=feature_cols,
            group_to_indices=pre_full.group_to_indices,
            fitness_cfg=config["fitness"],
        )
        baseline_row = {
            "algorithm": "baseline_rf_default",
            "seed": seed,
            "budget_b": 0,
            "evaluations_used": 0,
            "val_best_score": np.nan,
            "val_recall": np.nan,
            "val_fpr": np.nan,
            "test_score": baseline.score,
            "test_accuracy": baseline.metrics["accuracy"],
            "test_precision": baseline.metrics["precision"],
            "test_recall": baseline.metrics["recall"],
            "test_f1": baseline.metrics["f1"],
            "test_fpr": baseline.metrics["fpr"],
            "test_selected_features": baseline.metrics["selected_features"],
            "test_runtime_sec": baseline.metrics["runtime_sec"],
            "test_fit_time_sec": baseline.metrics["fit_time_sec"],
            "test_predict_time_sec": baseline.metrics["predict_time_sec"],
            "best_params_json": json.dumps(baseline.solution.params, sort_keys=True),
        }
        all_rows.append(baseline_row)
        _append_row_csv(incremental_path, baseline_row)

        for algorithm in ["ga", "pso", "sa"]:
            # New evaluator instance per (algorithm, seed) keeps cache isolation explicit.
            evaluator = ObjectiveEvaluator(
                algorithm_name=algorithm,
                seed=seed,
                x_train_inner=x_train_inner,
                y_train_inner=y_train_inner,
                x_val_inner=x_val_inner,
                y_val_inner=y_val_inner,
                original_features=feature_cols,
                group_to_indices=pre_inner.group_to_indices,
                search_space=search_space,
                k_min=k_min,
                fitness_cfg=config["fitness"],
                n_jobs=1,
            )

            opt_result = _run_single_optimizer(
                algorithm=algorithm,
                evaluator=evaluator,
                budget_b=budget_b,
                n_feature_genes=d_features,
                seed=seed,
                optimizer_cfg=config["optimizers"][algorithm],
            )
            if opt_result["best_solution"] is None:
                raise RuntimeError(f"{algorithm} produced no best solution.")

            _save_convergence(
                history=opt_result["history"],
                out_path=out_dirs["convergence"] / f"{algorithm}_seed_{seed}.csv",
                algorithm=algorithm,
                seed=seed,
            )

            final_test = evaluate_solution_on_test(
                solution=opt_result["best_solution"],
                x_train_full=x_train_full,
                y_train_full=full_y,
                x_test=x_test,
                y_test=test_y,
                original_features=feature_cols,
                group_to_indices=pre_full.group_to_indices,
                fitness_cfg=config["fitness"],
                seed=seed,
                n_jobs=1,
            )

            opt_row = {
                    "algorithm": algorithm,
                    "seed": seed,
                    "budget_b": budget_b,
                    "evaluations_used": opt_result["evaluations"],
                    "val_best_score": opt_result["best_score"],
                    "val_recall": opt_result["best_metrics"]["recall"],
                    "val_fpr": opt_result["best_metrics"]["fpr"],
                    "test_score": final_test.score,
                    "test_accuracy": final_test.metrics["accuracy"],
                    "test_precision": final_test.metrics["precision"],
                    "test_recall": final_test.metrics["recall"],
                    "test_f1": final_test.metrics["f1"],
                    "test_fpr": final_test.metrics["fpr"],
                    "test_selected_features": final_test.metrics["selected_features"],
                    "test_runtime_sec": final_test.metrics["runtime_sec"],
                    "test_fit_time_sec": final_test.metrics["fit_time_sec"],
                    "test_predict_time_sec": final_test.metrics["predict_time_sec"],
                    "best_params_json": json.dumps(
                        opt_result["best_solution"].params,
                        sort_keys=True,
                    ),
                }
            all_rows.append(opt_row)
            _append_row_csv(incremental_path, opt_row)

        seed_df = pd.DataFrame([r for r in all_rows if r["seed"] == seed])
        seed_df.to_csv(out_dirs["raw"] / f"seed_{seed}_results.csv", index=False)

    all_runs_df = pd.DataFrame(all_rows)
    all_runs_df.to_csv(out_dirs["raw"] / "all_runs.csv", index=False)
    summary_df = _aggregate_summary(all_runs_df, out_dirs["results"] / "summary.csv")

    if not skip_plots and bool(config["output"].get("save_plots", True)):
        _create_plots(
            all_runs=all_runs_df,
            convergence_dir=out_dirs["convergence"],
            plots_dir=out_dirs["plots"],
        )

    return all_runs_df, summary_df


## Run Experiment


In [ ]:

active_budget = SMOKE_BUDGET if RUN_SMOKE_TEST else None
active_max_seeds = SMOKE_MAX_SEEDS if RUN_SMOKE_TEST else None

all_runs_df, summary_df = run_coursework_experiment(
    project_root=PROJECT_ROOT,
    config=CONFIG,
    budget_override=active_budget,
    max_seeds=active_max_seeds,
    skip_plots=False,
    config_raw_text=yaml.safe_dump(CONFIG, sort_keys=False),
)

print('Finished. rows=', len(all_runs_df))
print('Results directory =', PROJECT_ROOT / CONFIG['output']['results_dir'])
display(summary_df)
